# Non-Conservative Fock Attention PARFLM — OpenWebText Scale-Up

## Motivation (comparison baseline)

This notebook trains the **non-conservative** `FockAttentionPARFLM`
(`parf/model_fock_attention.py`) — the model already validated on
TinyStories — on OpenWebText, as a head-to-head baseline for the
*conservative* context-mixing experiments
(`colab_fock_depthcond_vtheta_openwebtext.ipynb` and
`colab_xi_attention_openwebtext.ipynb`).

`FockAttentionPARFLM` implements the literal §5.1 "attention as virtual
particle exchange" Feynman diagram as a **non-conservative force** injected
after each conservative Verlet layer step:

> token j emits (key k_j, value v_j); token i absorbs with query q_i
> α_ij = softmax_{j≤i}( q_i · k_j / √d_k )
> F_exchange_i = Σ_j α_ij · v_j
> h_new ← h_new + (dt²/m) · tanh(scale) · F_exchange

The exchange force is **not** the gradient of a scalar potential — `q, k, v`
are projected from the *live* hidden state `h` (not detached ξ), so the
softmax routing depends on `h` and the induced force has a non-symmetric
Jacobian.  This is the deliberate point of comparison: it gives the model
full softmax-attention expressivity at the cost of breaking the conservative
(potential-derived force) structure.

## What this is vs. the conservative comparators

| Notebook | Context mixing | Conservative? | Registers? |
|---|---|---|---|
| `…_depthcond_vtheta_…` | sparse top-`k` `V_φ` pair potential | ✓ | ✓ (Fock) |
| `…_xi_attention_…` | dense xi-routed attention **potential** | ✓ | ✗ |
| **this** (`…_fock_attention_…`) | sparse `V_φ` **+** dense non-conservative exchange | ✗ | ✗ |

`FockAttentionPARFLM` keeps the conservative multi-xi `V_φ` pair potential
(via the inherited Verlet step) and **adds** the non-conservative exchange
force on top — so it is the "what does breaking conservativity buy us?"
control.  The learnable gate `tanh(exchange_scale)` starts at 0
(`EXCHANGE_SCALE_INIT = 0.0`), i.e. the model begins as the pure conservative
multi-xi PARF and learns how far to open the non-conservative channel.

## What is kept identical to the conservative runs

- Multi-context (+ optional depth-conditioned) Gaussian `V_θ` self-energy.
- Multi-resolution K-EMA ξ channels (`XI_OVERRIDE='5long'`).
- The sparse `V_φ` pair potential and its knobs (`TOP_K`, `V_PHI_*`).
- Velocity-Verlet integration, log-frequency mass, output bias, untied
  read-out head, WSD schedule, per-layer force scale, batch/steps/seed.

## What is removed

- The Fock register pool (creation/destruction gates): `FockAttentionPARFLM`
  has no registers — it is the λ=0 instantaneous-exchange limit.

## Added — exchange-force knobs

| Knob | Meaning |
|------|---------|
| `EXCHANGE_N_HEADS` | number of attention heads in the exchange force |
| `EXCHANGE_D_K` | per-head key/query/value dimension |
| `EXCHANGE_SCALE_INIT` | initial gate value (0 ⇒ force off at init, model opens it) |
| `EXCHANGE_INIT_SCALE` | weight init std of the exchange projections |

## Prerequisites

- OpenWebText tokenized and cached on Google Drive (reused from earlier phases).
- An H100 / Blackwell-class 80 GB+ GPU.


In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────

# ── V_theta architecture (self-energy; unchanged from conservative runs) ──
V_THETA_VARIANT        = 'gaussian'    # 'gaussian' | 'mlp'
W_SCALE                = 1.0
V_THETA_WELLS_PER_HEAD = 8             # Gaussian wells per multi-context head

# ── Depth-conditioning (option A: cheap per-layer V_theta untying) ──
#   True  -> depth-conditioned shared bank (tag 'dcvt...')
#   False -> plain multi-context bank      (tag 'mcvt...')
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02

# ── Xi channel override (drives V_theta context AND the K-EMA ξ that V_φ /
#    the exchange routing operate over) ──────────────────────────────
XI_OVERRIDE     = '5long'       # '5long' (default) | 5 | 6 | '4long' | None

_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS_CFG:
    XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)

V_THETA_N_HEADS = XI_CHANNELS    # one V_theta bank per xi channel (default)

# ── PARF V_phi knobs (conservative pair potential, kept by this model) ──
V_PHI_KIND      = 'structural_competitive'  # 'structural_competitive' | 'structural' | 'mlp'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16            # sparse pairs per query
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32            # type-vector dimension d_l
V_PHI_D_ANGLE   = 16            # value-angle dimension K

# ── Non-conservative exchange force (the §5.1 Feynman diagram) ────
# q,k,v are projected from the LIVE hidden state h (not detached ξ), so the
# softmax routing depends on h and the induced force is NOT -∇ of a scalar.
# tanh(EXCHANGE_SCALE_INIT) gates the force; 0.0 starts it off so the model
# begins as the pure conservative multi-xi PARF and learns to open it.
EXCHANGE_N_HEADS    = 4
EXCHANGE_D_K        = 48        # per-head key/query/value dim
EXCHANGE_SCALE_INIT = 0.0       # tanh-gate init (0 -> exchange off at start)
EXCHANGE_INIT_SCALE = 0.02      # weight init std of exchange projections

# ── Output read-out head (D0.4 long-tail fix) ────────────────────
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False         # untied W_out for this experiment

# ── Optimizer ─────────────────────────────────────────────────────
OPTIMIZER = 'adamw'             # 'adamw' | 'lamb' | 'lion'
GRAD_CENTRALIZATION = False

# ── LR schedule ──────────────────────────────────────────────────
LR_SCHEDULE     = 'wsd'         # 'cosine' | 'wsd'
WSD_WARMUP_FRAC = 0.05
WSD_STABLE_FRAC = 0.60
WSD_LR_FLOOR    = None          # default = LR * 0.05 (resolved in Cell 5)

# ── Batch / accumulation ─────────────────────────────────────────
GRAD_ACCUM      = 1             # BATCH_SIZE is auto-probed for OOM safety.

# ── Variant tag ──────────────────────────────────────────────────
_variant_parts = []
if V_THETA_VARIANT == 'mlp':
    _variant_parts.append('mlp_vtheta')
elif V_THETA_VARIANT == 'gaussian' and V_THETA_WELLS_PER_HEAD != 8:
    _variant_parts.append(f'k{V_THETA_WELLS_PER_HEAD}')
if V_PHI_KIND == 'mlp':
    _variant_parts.append(f'mlp_vphi_h{V_PHI_MLP_HIDDEN}')
elif V_PHI_KIND == 'structural':
    _variant_parts.append('struct_vphi')
if XI_OVERRIDE is not None:
    _variant_parts.append(f'xi{XI_OVERRIDE}')
if TOP_K != 8:
    _variant_parts.append(f'topk{TOP_K}')
if V_PHI_D_TYPE != 16 or V_PHI_D_ANGLE != 8:
    _variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
if V_PHI_N_HEADS != 1:
    _variant_parts.append(f'mh{V_PHI_N_HEADS}')
# Non-conservative exchange descriptor.
_variant_parts.append(f'fockattn{EXCHANGE_N_HEADS}')
if V_THETA_N_HEADS > 1:
    if V_THETA_DEPTH_CONDITION:
        _variant_parts.append(f'dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
    else:
        _variant_parts.append(f'mcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
if USE_OUTPUT_BIAS:
    _variant_parts.append('ob')
if not TIE_EMBEDDINGS:
    _variant_parts.append('untied')
if OPTIMIZER != 'adamw':
    _variant_parts.append(OPTIMIZER)
if GRAD_CENTRALIZATION:
    _variant_parts.append('gc')
if LR_SCHEDULE != 'cosine':
    _variant_parts.append(LR_SCHEDULE)
_variant_tag = '_'.join(_variant_parts)

print(f'Config: model=FockAttentionPARFLM (NON-conservative exchange)')
print(f'  V_theta={V_THETA_VARIANT}')
if V_THETA_VARIANT == 'gaussian':
    print(f'  wells_per_head={V_THETA_WELLS_PER_HEAD}, w_scale={W_SCALE}')
if V_THETA_N_HEADS > 1:
    print(f'  multi-context V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total attractors')
    if V_THETA_DEPTH_CONDITION:
        print(f'  depth-conditioned: shared bank + per-layer codes '
              f'(init_std={V_THETA_DEPTH_CODE_INIT_STD})')
print(f'  V_phi={V_PHI_KIND}'
      + (f' (mlp_hidden={V_PHI_MLP_HIDDEN})' if V_PHI_KIND == 'mlp' else ''))
print(f'  exchange (non-conservative): heads={EXCHANGE_N_HEADS}  d_k={EXCHANGE_D_K}  '
      f'gate_init=tanh({EXCHANGE_SCALE_INIT})={__import__("math").tanh(EXCHANGE_SCALE_INIT):.3f}')
if XI_OVERRIDE is not None:
    print(f'  XI_OVERRIDE={XI_OVERRIDE} -> {XI_CHANNELS}ch, '
          f'horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  PARF top_k={TOP_K}, V_phi heads={V_PHI_N_HEADS}, '
      f'd_type={V_PHI_D_TYPE} (eff {V_PHI_N_HEADS*V_PHI_D_TYPE}), '
      f'd_angle={V_PHI_D_ANGLE} (eff {V_PHI_N_HEADS*V_PHI_D_ANGLE})')
print(f'  read-out: output_bias={USE_OUTPUT_BIAS}, '
      + ('tied E^T' if TIE_EMBEDDINGS else 'UNTIED W_out'))
print(f'  optimizer={OPTIMIZER}  grad_centralization={GRAD_CENTRALIZATION}')
print(f'  LR schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
if _variant_tag:
    print(f'  [variant] tag={_variant_tag}')


In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_attention_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'fock_attention' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fockattn_owt' + (f'_{_variant_tag}' if _variant_tag else '')
TOTAL_STEPS   = 100_000
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

import glob as _glob, re as _re
_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) — resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'\nResuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found — training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# ── Cell 3: Data loading (reuse cached OpenWebText) ──────────────
from data_module import get_batch

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt',
    'semsimula_xi_attention_owt',
    'semsimula_fock_multicontext_vtheta_owt',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_attention_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4b: PMI Spectral Diversity — K_MIX estimator ──────────────
RANK_EFF_TINYSTORIES = None


def compute_pmi_effective_rank(token_ids, vocab_size, top_v=8192, window=5, n_components=512):
    """Compute effective spectral rank of the PMI matrix (Roy-Vetterli)."""
    print(f'  Building co-occurrence matrix (top_v={top_v}, window={window}) ...')
    token_counts = np.bincount(token_ids.astype(np.int64), minlength=vocab_size)
    top_v_ids = np.argsort(-token_counts)[:top_v]
    id_to_local = np.full(vocab_size, -1, dtype=np.int64)
    id_to_local[top_v_ids] = np.arange(top_v)

    cooc = np.zeros((top_v, top_v), dtype=np.float64)
    local_ids = id_to_local[token_ids.astype(np.int64)]
    for offset in range(1, window + 1):
        a, b = local_ids[:-offset], local_ids[offset:]
        valid = (a >= 0) & (b >= 0)
        np.add.at(cooc, (a[valid], b[valid]), 1.0)
    cooc = cooc + cooc.T

    row_sums = cooc.sum(axis=1, keepdims=True)
    total = cooc.sum()
    expected = row_sums * row_sums.T / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log(cooc / np.maximum(expected, 1e-12))
    pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)
    np.fill_diagonal(pmi, 0.0)

    print(f'  Computing truncated SVD (k={n_components}) ...')
    from scipy.sparse.linalg import svds
    from scipy.sparse import csr_matrix
    pmi_sparse = csr_matrix(pmi)
    _, s, _ = svds(pmi_sparse, k=min(n_components, top_v - 1))
    s = np.sort(s)[::-1]

    s_norm = s / s.sum()
    s_norm = s_norm[s_norm > 1e-12]
    entropy = -np.sum(s_norm * np.log(s_norm))
    rank_eff = np.exp(entropy)
    return rank_eff, s


#print('Computing PMI spectral diversity for OpenWebText ...')
#t_pmi = time.time()
#rank_eff_owt, sv_owt = compute_pmi_effective_rank(train_ids, VOCAB_SIZE)
#print(f'  Done in {time.time() - t_pmi:.1f}s')
#print(f'\n  rank_eff(OpenWebText) = {rank_eff_owt:.1f}')
#print(f'  Top-5 singular values: {sv_owt[:5]}')
#print(f'  SV decay ratio (s[0]/s[50]): {sv_owt[0]/sv_owt[min(50, len(sv_owt)-1)]:.1f}')
#
#if RANK_EFF_TINYSTORIES is not None:
#    diversity_ratio = rank_eff_owt / RANK_EFF_TINYSTORIES
#    suggested_k = max(8, round(8 * diversity_ratio))
#    print(f'\n  rank_eff(TinyStories) = {RANK_EFF_TINYSTORIES:.1f}  (reference)')
#    print(f'  Diversity ratio: {diversity_ratio:.2f}x')
#    print(f'  Suggested V_THETA_WELLS_PER_HEAD = {suggested_k}')
#else:
#    print(f'\n  [info] RANK_EFF_TINYSTORIES not set.')
#    print(f'  Set it at the top of this cell for scaling guidance.')

In [ ]:
# ── Cell 4: Model config + non-conservative Fock attention ───────
import math, types
from model_fock_attention import FockAttentionPARFLM, FockAttentionConfig
import model_parf_multixi
import model_parf

_XI_PRESETS = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS:
    XI_ALPHA_INITS = _XI_PRESETS[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)
print(f'Xi: {XI_CHANNELS} channels, alphas={XI_ALPHA_INITS}')
print(f'  Horizons: ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tokens')

LAMBDA_V       = 1e-2
BLOCK_SIZE     = 512

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

# (d, L) architecture tiers — FockAttentionPARFLM has no Fock registers.
ARCH_TIERS = [
    (384, 16),
    (384, 12),
    (256, 16),
    (256,  8),
]


def make_config(d, L):
    return FockAttentionConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        # ── non-conservative exchange force ──
        exchange_n_heads=EXCHANGE_N_HEADS,
        exchange_d_k=EXCHANGE_D_K,
        exchange_scale_init=EXCHANGE_SCALE_INIT,
        exchange_init_scale=EXCHANGE_INIT_SCALE,
    )


def install_depth_routing_baseline(model):
    \"\"\"Broadcast the active layer index to a depth-conditioned V_theta.

    FockAttentionPARFLM has no _fock_layer_step; its conservative dynamics run
    through _baseline_layer_step(..., layer_idx).  Patch that to set the active
    layer before super()._layer_step evaluates V_theta.  Safe under gradient
    checkpointing (attribute is set inside the same call that consumes it).
    \"\"\"
    from model_gaussian_vtheta import DepthConditionedMultiContextGaussianVTheta
    vt = getattr(model, 'V_theta', None)
    if not isinstance(vt, DepthConditionedMultiContextGaussianVTheta):
        return
    if getattr(model, '_depth_routing_installed', False):
        return
    _orig = model._baseline_layer_step.__func__

    def _routed(self, h, h_prev, m_b, gamma, dt, layer_idx, *a, **kw):
        v = getattr(self, 'V_theta', None)
        if isinstance(v, DepthConditionedMultiContextGaussianVTheta):
            v.set_active_layer(layer_idx)
        return _orig(self, h, h_prev, m_b, gamma, dt, layer_idx, *a, **kw)

    model._baseline_layer_step = types.MethodType(_routed, model)
    model._depth_routing_installed = True


def build_structured_vtheta(model, d, variant, device):
    \"\"\"Swap model.V_theta with a structured Gaussian variant (same as the
    conservative runs).  Depth routing is installed via the baseline-layer
    patch above.\"\"\"
    if variant == 'mlp':
        return
    if variant != 'gaussian':
        raise ValueError(f'This notebook supports gaussian|mlp V_theta; got {variant!r}')
    xi_d = XI_CHANNELS * d
    from model_gaussian_vtheta import (
        MixtureGaussianVTheta, GaussianVThetaMultiXiAdapter,
        MultiContextGaussianVTheta, DepthConditionedMultiContextGaussianVTheta,
    )
    _init_log_prec = -math.log(d)
    _prec_max = 2.0 / d
    _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
    _sigma_min = 1.0 / (_prec_max ** 0.5)

    if V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION:
        model.V_theta = DepthConditionedMultiContextGaussianVTheta(
            d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
            n_layers=model.cfg.L, w_scale=W_SCALE,
            init_log_precision=_init_log_prec, precision_max=_prec_max,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        ).to(device)
        install_depth_routing_baseline(model)
        _n_code = model.V_theta.depth_code.numel()
        print(f'V_theta -> DepthConditionedMultiContextGaussian('
              f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells, '
              f'L={model.cfg.L}, code_params={_n_code:,}, '
              f'sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')
        print(f'  depth routing installed on _baseline_layer_step')
    elif V_THETA_N_HEADS > 1:
        model.V_theta = MultiContextGaussianVTheta(
            d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS, w_scale=W_SCALE,
            init_log_precision=_init_log_prec, precision_max=_prec_max,
        ).to(device)
        print(f'V_theta -> MultiContextGaussian({V_THETA_N_HEADS} heads x '
              f'{V_THETA_WELLS_PER_HEAD} wells = {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total, '
              f'sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')
    else:
        inner = MixtureGaussianVTheta(
            d=d, K=V_THETA_WELLS_PER_HEAD, w_scale=W_SCALE, xi_d=xi_d,
            init_log_precision=_init_log_prec, precision_max=_prec_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)
        print(f'V_theta -> ConcatGaussian(K={V_THETA_WELLS_PER_HEAD}, '
              f'sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')


# ── Try architecture tiers ─────────────────────────────────────────
model = None
model_cfg = None
for d, L in ARCH_TIERS:
    try:
        cfg = make_config(d, L)
        mdl = FockAttentionPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())
        build_structured_vtheta(mdl, d, V_THETA_VARIANT, DEVICE)
        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        n_ex = mdl.get_exchange_overhead()
        _vt_label = {'mlp': 'MLP', 'gaussian': 'Gaussian'}[V_THETA_VARIANT]
        _mc_label = f' ({V_THETA_N_HEADS} heads)' if V_THETA_N_HEADS > 1 else ' (concat)'
        print(f'Trying d={d} L={L} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} {_vt_label}{_mc_label}, '
              f'exchange {n_ex:,})')
        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} — trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# ── Auto batch size (GRAD_ACCUM is fixed from Cell 0) ──────────────
BATCH_SIZE = 4
if DEVICE == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _probe_sizes = [16, 12, 8, 6, 4] if _vram_gb >= 70 else [12, 8, 6, 4]
    for bs in _probe_sizes:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_ex = model.get_exchange_overhead()
IS_STRUCTURED = V_THETA_VARIANT == 'gaussian'

_vtheta_name = {'gaussian': 'Gaussian', 'mlp': 'MLP'}[V_THETA_VARIANT]
print(f'\nModel: FockAttentionPARFLM (non-conservative) + {_vtheta_name} V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,}, exchange: {n_ex:,})')
print(f'  d={model_cfg.d}  L={model_cfg.L}')
print(f'  exchange: heads={EXCHANGE_N_HEADS}  d_k={EXCHANGE_D_K}  '
      f'gate=tanh({float(model.exchange_scale):.3f})={math.tanh(float(model.exchange_scale)):.3f}')
if V_THETA_N_HEADS > 1:
    print(f'  V_theta heads: {V_THETA_N_HEADS} x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total attractors'
          + ('  (depth-conditioned)' if V_THETA_DEPTH_CONDITION else ''))
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  lambda_V={LAMBDA_V}  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  IS_STRUCTURED={IS_STRUCTURED}')


In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

LR            = 3e-4      # peak LR (WSD keeps it here longer)
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3
EVAL_INTERVAL = 500
EVAL_ITERS    = 40
LOG_INTERVAL  = 50
SEED          = 0

# Resolve WSD_LR_FLOOR now that LR is final.
if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    \"\"\"Unified LR schedule supporting cosine and WSD.\"\"\"
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'exchange_n_heads': EXCHANGE_N_HEADS, 'exchange_d_k': EXCHANGE_D_K,
            'exchange_scale_init': EXCHANGE_SCALE_INIT,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'exchange_scale': float(torch.tanh(model.exchange_scale)),
        'variant': f'fock_attention_{V_THETA_VARIANT}_ex{EXCHANGE_N_HEADS}',
        'corpus': 'openwebtext',
        'phase': 7,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# ── Optimizer ──
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')

# ── Resume ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training state ──
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]

def _log_write(record_str):
    \"\"\"Write a JSONL record to the Drive log, remounting on transport error.\"\"\"
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost, training continues.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}; training continues.')
                return

log_f = _log_fh

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        print(f'No canonical _best.pt; using {_best_ckpt_path.name}')
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical
        print(f'  Copied to canonical: {_canonical.name}')

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0

def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s

steps_this_session = 0

# ── Schedule summary ──
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'Non-conservative Fock attention ({_vtheta_name} V_theta): '
      f'steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}  grad_clip_vphi={GRAD_CLIP_VPHI}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}')
print(f'  exchange: heads={EXCHANGE_N_HEADS} d_k={EXCHANGE_D_K}  params={n_params:,}')
if V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION:
    print(f'  V_theta depth-conditioned: shared bank + per-layer codes')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
print(f'{"="*60}\n')

for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    if GRAD_CENTRALIZATION:
        for p in model.parameters():
            if p.grad is not None and p.grad.dim() >= 2:
                p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

    nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
    grad_norm = nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        if IS_STRUCTURED and V_THETA_N_HEADS == 1:
            if hasattr(getattr(model.V_theta, 'inner', None), 'clamp_params'):
                model.V_theta.inner.clamp_params()
        elif IS_STRUCTURED and V_THETA_N_HEADS > 1:
            for bank in model.V_theta.banks:
                if hasattr(bank, 'clamp_params'):
                    bank.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # ── Watchdog ──
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        ex_scale = float(torch.tanh(model.exchange_scale))
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  gamma={model.gamma.item():.3f}  '
            f'ex_scale={ex_scale:.3f}  alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        _log_write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'exchange_scale': ex_scale,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        _log_write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

_log_fh[0].close()
print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')